# Notebook 03 - Analiza Multivariata

**Proiect:** Modelarea si prognoza volatilitatii S&P 500 sub influenta sentimentului din stiri financiare

**Disciplina:** Analiza avansata a seriilor de timp si previziune

---

## Structura

| Sectiune | Continut |
|---|---|
| 4.1 | Specificarea problemei multivariate si variabilele folosite |
| 4.2 | Pregatirea datelor (demean sentiment, log VIX, aliniere) |
| 4.3 | Teste preliminare multivariate (cointegrare Johansen) |
| 4.4 | VAR - selectie lag, estimare, diagnostic |
| 4.5 | Granger causality (4 directii) |
| 4.6 | IRF + FEVD |
| 4.7 | DCC-GARCH bivariat |
| 4.8 | GARCH-X (modelul central) |
| 4.9 | Comparatie finala cu benchmark univariat |

> **Nota metodologica:** Notebook-ul 02 a stabilit benchmark-ul univariat.
> Aici adaugam sentimentul ca variabila explicativa si testam daca aduce
> informatie incrementala in prognoza volatilitatii S&P 500.


---
## Setup si incarcare date

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Serii de timp
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

# ARCH/GARCH
from arch import arch_model
from arch.unitroot import PhillipsPerron

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.figsize': (13, 5),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

DATA_PATH = Path('./data/processed/sp500_sentiment_daily.parquet')
FIG_PATH  = Path('./data/figures')
FIG_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
df.index = pd.to_datetime(df.index)
df.index.freq = None

print(f"Dataset: {df.shape}")
print(f"Perioada: {df.index.min().date()} -> {df.index.max().date()}")
print(f"Coloane disponibile: {list(df.columns)}")


---
## 4.1 Specificarea problemei multivariate si variabilele folosite

### Intrebarea centrala de cercetare

> *Aduce sentimentul din stirile financiare NYT informatie incrementala
> fata de modelele GARCH clasice in prognoza volatilitatii S&P 500?*

### De ce analiza multivariata

Modelele univariate (Notebook 02) folosesc doar istoricul lui r_t pentru a prezice
volatilitatea. Dar piata financiara e influentata si de **informatia externa** --
stirile, anunturile Fed, evenimentele geopolitice. Sentimentul din presa financiara
captureaza tocmai aceasta informatie externa.

### Variabilele sistemului

| Variabila | Sursa | Rol |
|---|---|---|
| r_t | Yahoo Finance | Variabila tinta primara |
| r_t_sq | Derivata din r_t | Proxy volatilitate realizata |
| S_polarity_dm | NYT + FinBERT | Sentiment demeaned -- variabila cheie |
| log_vix | Yahoo Finance | Variabila de control (volatilitate implicita) |
| n_articles | NYT Archive API | Proxy pentru atentia media |
| S_polarity_lag1 | Derivata din S_polarity | Sentiment laguit pentru GARCH-X |

### De ce demeaned pentru sentiment

S_polarity are media -0.34 (bias negativ structural al presei financiare).
In modele multivariate, interpretam coeficientii ca efectul unei deviatii
fata de nivelul obisnuit de sentiment, nu fata de zero.
Demeaning: S_polarity_dm = S_polarity - mean(S_polarity_train)

### Split train/test (identic cu Notebook 02)

```
Train: 2018-01-02 -> 2023-12-29
Test:  2024-01-02 -> 2024-12-30
```


---
## 4.2 Pregatirea datelor

In [ ]:
# ── Split ────────────────────────────────────────────────────────────
TRAIN_END  = '2023-12-31'
TEST_START = '2024-01-01'

train = df[df.index <= TRAIN_END].copy()
test  = df[df.index >= TEST_START].copy()

print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} obs)")
print(f"Test:  {test.index.min().date()}  -> {test.index.max().date()}   ({len(test)} obs)")

# ── Variabile de baza ─────────────────────────────────────────────────
r_train = train['r_t'].dropna()
r_test  = test['r_t'].dropna()

# ── Demean sentiment pe media din TRAIN ──────────────────────────────
s_mean_train = train['S_polarity'].mean()
print(f"Media S_polarity pe train: {s_mean_train:.4f}")
print("Demeaning: S_polarity_dm = S_polarity - {:.4f}".format(s_mean_train))

df['S_polarity_dm'] = df['S_polarity'] - s_mean_train
train['S_polarity_dm'] = train['S_polarity'] - s_mean_train
test['S_polarity_dm']  = test['S_polarity']  - s_mean_train

# ── Log VIX ───────────────────────────────────────────────────────────
df['log_vix']    = np.log(df['vix'])
train['log_vix'] = np.log(train['vix'])
test['log_vix']  = np.log(test['vix'])

# ── Matricea multivariata pentru VAR ─────────────────────────────────
# Variabile endogene: r_t, r_t_sq, S_polarity_dm
# Variabila exogena:  log_vix
var_cols = ['r_t', 'r_t_sq', 'S_polarity_dm']

var_train = train[var_cols].dropna()
var_test  = test[var_cols].dropna()

print(f"Matricea VAR train: {var_train.shape}")
print(f"Matriea VAR test:  {var_test.shape}")
print(f"Statistici descriptive variabile multivariate (train):")
print(var_train.describe().round(4))


In [ ]:
# ── Vizualizare variabile multivariate ───────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(train.index, train['r_t'], color='steelblue', linewidth=0.7)
axes[0].axhline(0, color='black', linewidth=0.4, linestyle='--')
axes[0].set_title('Log-randamente r_t (%) -- Train')
axes[0].set_ylabel('%')

axes[1].plot(train.index, train['r_t_sq'], color='crimson', linewidth=0.6)
axes[1].set_title('Volatilitate proxy r_t^2 -- Train')
axes[1].set_ylabel('r_t^2')

axes[2].plot(train.index, train['S_polarity_dm'], color='forestgreen', linewidth=0.6)
axes[2].axhline(0, color='black', linewidth=0.4, linestyle='--')
axes[2].set_title('Sentiment S_polarity demeaned -- Train')
axes[2].set_ylabel('Polarity (demeaned)')

for ax in axes:
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-04-15'),
               alpha=0.12, color='red')
    ax.axvspan(pd.Timestamp('2022-02-24'), pd.Timestamp('2022-06-30'),
               alpha=0.12, color='orange')

plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_01_var_series.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 4.3 Teste preliminare multivariate

### Stationaritate individuala

Confirmam stationaritatea fiecarei variabile pe setul de train.
(Deja verificat in Notebook 01, repetam pe subsampleul de train.)

### Testul de cointegrare Johansen

Testul Johansen verifica daca exista o **relatie de echilibru pe termen lung**
intre variabilele sistemului.

H0: numarul de vectori de cointegrare este <= r

Daca variabilele sunt cointegrare, trebuie sa folosim VECM in loc de VAR.
In practica, pe date financiare zilnice, cointegrarea e rara -- ne asteptam
sa nu gasim cointegrare si sa folosim VAR pe niveluri (seriile sunt deja stationare).


In [ ]:
# ── Stationaritate pe train ───────────────────────────────────────────
def adf_test(series, name):
    s = series.dropna()
    stat, p, *_ = adfuller(s, autolag='AIC')
    result = 'STATIONAR' if p < 0.05 else 'NON-STATIONAR'
    print(f"  {name:<25}  ADF stat={stat:8.3f}  p={p:.4f}  -> {result}")

print("=== STATIONARITATE (ADF) -- setul TRAIN ===")
adf_test(train['r_t'],           'r_t')
adf_test(train['r_t_sq'],        'r_t_sq')
adf_test(train['S_polarity_dm'], 'S_polarity_dm')
adf_test(train['log_vix'],       'log_vix')
adf_test(train['n_articles'],    'n_articles')


In [ ]:
# ── Test Johansen ────────────────────────────────────────────────────
print("=== TEST COINTEGRARE JOHANSEN ===")
print("Variabile: r_t, r_t_sq, S_polarity_dm")
print()

data_johansen = var_train.dropna().values

try:
    result_joh = coint_johansen(data_johansen, det_order=0, k_ar_diff=1)

    print("Trace statistic:")
    print(f"{'r':<5} {'Trace stat':>12} {'Val. critica 5%':>16} {'Respingem H0?':>15}")
    print("-" * 52)
    for i in range(len(result_joh.lr1)):
        trace_stat = result_joh.lr1[i]
        crit_val   = result_joh.cvt[i, 1]  # 5% critical value
        reject     = trace_stat > crit_val
        print(f"  {i:<3}  {trace_stat:12.3f}  {crit_val:16.3f}  {'DA' if reject else 'NU':>15}")

    print()
    print("Max eigenvalue statistic:")
    print(f"{'r':<5} {'Max stat':>12} {'Val. critica 5%':>16} {'Respingem H0?':>15}")
    print("-" * 52)
    for i in range(len(result_joh.lr2)):
        max_stat = result_joh.lr2[i]
        crit_val = result_joh.cvm[i, 1]
        reject   = max_stat > crit_val
        print(f"  {i:<3}  {max_stat:12.3f}  {crit_val:16.3f}  {'DA' if reject else 'NU':>15}")

    print()
    print("Concluzie: daca nu respingem H0 pentru r=0 -> nu exista cointegrare -> folosim VAR")
    print("Concluzie: daca respingem H0 pentru r=0 -> exista cointegrare -> am folosi VECM")

except Exception as e:
    print(f"Johansen a esuat: {e}")
    print("Continuam cu VAR -- pe serii stationare VAR e intotdeauna valid.")


---
## 4.4 VAR (Vector Autoregression)

### Ce este VAR

VAR modeleaza simultan mai multe serii de timp, permitand fiecareia sa fie
influentata de valorile trecute ale tuturor celorlalte.

Pentru sistemul nostru [r_t, r_t_sq, S_polarity_dm]:

```
r_t       = c1 + sum_k(A1k * r_{t-k})     + sum_k(A2k * r_t_sq_{t-k}) + sum_k(A3k * S_{t-k}) + e1_t
r_t_sq    = c2 + sum_k(B1k * r_{t-k})     + sum_k(B2k * r_t_sq_{t-k}) + sum_k(B3k * S_{t-k}) + e2_t
S_polarity= c3 + sum_k(C1k * r_{t-k})     + sum_k(C2k * r_t_sq_{t-k}) + sum_k(C3k * S_{t-k}) + e3_t
```

### Selectie lag

Selectam numarul de lag-uri prin AIC, BIC, HQIC pe setul de train.


In [ ]:
# ── Selectie lag VAR ─────────────────────────────────────────────────
data_var = var_train.dropna()

var_selector = VAR(data_var)
lag_results  = var_selector.select_order(maxlags=10)
print("=== SELECTIE LAG VAR ===")
print(lag_results.summary())

# Lag optim dupa AIC
p_aic = lag_results.aic
p_bic = lag_results.bic
p_opt = min(p_aic, 5)  # cap la 5 pentru parsimonie
print(f"Lag optim AIC: {p_aic}")
print(f"Lag optim BIC: {p_bic}")
print(f"Lag ales (max 5): {p_opt}")


In [ ]:
# ── Estimare VAR ─────────────────────────────────────────────────────
var_model = VAR(data_var)
var_fit   = var_model.fit(p_opt)
print(var_fit.summary())


In [ ]:
# ── Diagnostic reziduuri VAR ─────────────────────────────────────────
print("=== DIAGNOSTIC REZIDUURI VAR ===")

# Portmanteau test (echivalent Ljung-Box multivariat)
port_test = var_fit.test_whiteness(nlags=10)
print(port_test.summary())


In [ ]:
# ── Coeficienti cheie -- impactul sentimentului ───────────────────────
print("=== COEFICIENTI SENTIMENT -> R_T_SQ (ecuatia volatilitatii) ===")

# Extragem coeficientii pentru ecuatia r_t_sq
eq_idx = var_train.columns.tolist().index('r_t_sq')
coef_df = pd.DataFrame(
    var_fit.coefs.reshape(p_opt, len(var_cols), len(var_cols))[:, eq_idx, :],
    columns=var_cols,
    index=[f'lag_{i+1}' for i in range(p_opt)]
)
print("Coeficienti in ecuatia r_t_sq (volatilitate):")
print(coef_df.round(4))
print()
print("Interpretare: coloana S_polarity_dm arata efectul sentimentului")
print("la fiecare lag asupra volatilitatii zilnice.")


---
## 4.5 Granger Causality

### Ce testeaza Granger

Testul Granger raspunde la intrebarea:
*Valorile trecute ale lui X ajuta la prezicerea lui Y, peste ce stim deja din Y?*

**Atentie:** cauzalitate Granger NU inseamna cauzalitate reala -- inseamna
**predictibilitate temporala**. Daca S_t Granger-cauzeaza r_t_sq, inseamna
ca sentimentul de ieri prezice volatilitatea de azi.

### Cele 4 directii testate

| Test | Intrebarea |
|---|---|
| S -> r_t | Sentimentul prezice randamentele? |
| S -> r_t_sq | Sentimentul prezice volatilitatea? (CEL MAI IMPORTANT) |
| r_t -> S | Piata prezice sentimentul? |
| r_t_sq -> S | Volatilitatea prezice sentimentul? |

### Rezultat asteptat

Pe baza cross-correlatiei din Notebook 01 (~-0.08 la lag 1),
ne asteptam ca r_t_sq -> S sa fie mai puternic decat S -> r_t_sq.
Piata misca stirile mai mult decat stirile misca piata.

Daca gasim S -> r_t_sq semnificativ (chiar si marginal),
avem justificare empirica pentru GARCH-X.


In [ ]:
# ── Granger Causality Tests ───────────────────────────────────────────
max_lags = p_opt  # folosim acelasi lag ca VAR

print("=" * 65)
print("TEST 1: S_polarity_dm -> r_t (sentimentul prezice randamentele?)")
print("=" * 65)
gc1 = grangercausalitytests(
    data_var[['r_t', 'S_polarity_dm']],
    maxlag=max_lags, verbose=True
)


In [ ]:
print("=" * 65)
print("TEST 2: S_polarity_dm -> r_t_sq (sentimentul prezice VOLATILITATEA?)")
print("=" * 65)
gc2 = grangercausalitytests(
    data_var[['r_t_sq', 'S_polarity_dm']],
    maxlag=max_lags, verbose=True
)


In [ ]:
print("=" * 65)
print("TEST 3: r_t -> S_polarity_dm (piata prezice sentimentul?)")
print("=" * 65)
gc3 = grangercausalitytests(
    data_var[['S_polarity_dm', 'r_t']],
    maxlag=max_lags, verbose=True
)


In [ ]:
print("=" * 65)
print("TEST 4: r_t_sq -> S_polarity_dm (volatilitatea prezice sentimentul?)")
print("=" * 65)
gc4 = grangercausalitytests(
    data_var[['S_polarity_dm', 'r_t_sq']],
    maxlag=max_lags, verbose=True
)


In [ ]:
# ── Tabel sintetic Granger ────────────────────────────────────────────
print("=== TABEL SINTETIC GRANGER CAUSALITY ===")

def extract_granger_pval(gc_result, lag):
    try:
        return round(gc_result[lag][0]['ssr_ftest'][1], 4)
    except:
        return np.nan

rows = []
tests = [
    ('S -> r_t',      gc1, 'Sentimentul prezice randamentele?'),
    ('S -> r_t_sq',   gc2, 'Sentimentul prezice volatilitatea?'),
    ('r_t -> S',      gc3, 'Piata prezice sentimentul?'),
    ('r_t_sq -> S',   gc4, 'Volatilitatea prezice sentimentul?'),
]

for name, gc, question in tests:
    p_vals = [extract_granger_pval(gc, lag) for lag in range(1, max_lags+1)]
    min_p  = min([p for p in p_vals if not np.isnan(p)], default=np.nan)
    sig    = 'DA **' if min_p < 0.05 else ('Marginal *' if min_p < 0.10 else 'Nu')
    rows.append({'Test': name, 'p-value min': min_p,
                 'Semnificativ?': sig, 'Intrebare': question})

granger_df = pd.DataFrame(rows)
print(granger_df.to_string(index=False))
granger_df.to_csv(FIG_PATH / 'nb03_granger_results.csv', index=False)
print("Salvat: nb03_granger_results.csv")


---
## 4.6 Impulse Response Functions (IRF) si FEVD

### IRF

IRF arata cum raspunde o variabila la un soc de 1 deviatie standard
intr-o alta variabila, in urmatorii N pasi de timp.

Intrebarea cheie: *Daca sentimentul scade brusc cu 1 sigma azi,
cum evolueaza volatilitatea S&P in urmatoarele 10 zile?*

Ne asteptam: un soc negativ in sentiment -> crestere a volatilitatii
in urmatoarele 1-3 zile, cu revenire la normal dupa ~5 zile.

### FEVD (Forecast Error Variance Decomposition)

FEVD arata ce procent din variatia prognozei unei variabile
se datoreaza socurilor in fiecare variabila a sistemului.

Intrebarea cheie: *Cat din variatia volatilitatii (r_t_sq) se explica
prin socuri in sentiment vs. socuri proprii?*


In [ ]:
# ── IRF ──────────────────────────────────────────────────────────────
irf = var_fit.irf(periods=15)

# Plot IRF: rspuns r_t_sq la soc in S_polarity_dm
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Identificam indeksii variabilelor
idx_rt2 = var_cols.index('r_t_sq')
idx_s   = var_cols.index('S_polarity_dm')
idx_rt  = var_cols.index('r_t')

# IRF: r_t_sq la soc in S_polarity_dm
irf_vals = irf.irfs[:, idx_rt2, idx_s]
periods  = np.arange(len(irf_vals))

axes[0].bar(periods, irf_vals, color=['crimson' if v < 0 else 'steelblue'
                                       for v in irf_vals])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('IRF: Raspuns r_t^2 la soc in S_polarity_dm')
axes[0].set_xlabel('Zile dupa soc')
axes[0].set_ylabel('Raspuns volatilitate')

# IRF: r_t la soc in S_polarity_dm
irf_rt = irf.irfs[:, idx_rt, idx_s]
axes[1].bar(periods, irf_rt, color=['crimson' if v < 0 else 'steelblue'
                                     for v in irf_rt])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('IRF: Raspuns r_t la soc in S_polarity_dm')
axes[1].set_xlabel('Zile dupa soc')
axes[1].set_ylabel('Raspuns randamente')

plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_02_irf.png', dpi=130, bbox_inches='tight')
plt.show()

print("Interpretare:")
print("  - Bare negative: socul in sentiment reduce randamentele/creste volatilitatea")
print("  - Bare pozitive: socul in sentiment creste randamentele/reduce volatilitatea")
print("  - Revenire la zero: socul e tranzitoriu (cateva zile)")


In [ ]:
# ── FEVD ─────────────────────────────────────────────────────────────
fevd = var_fit.fevd(periods=15)

fig, ax = plt.subplots(figsize=(10, 5))

# FEVD pentru r_t_sq (variabila de interes: volatilitate)
fevd_rt2 = fevd.decomp[:, idx_rt2, :]  # shape: (periods, n_vars)
periods_f = np.arange(1, len(fevd_rt2) + 1)

bottom = np.zeros(len(fevd_rt2))
colors = ['steelblue', 'crimson', 'forestgreen']
for i, (col, color) in enumerate(zip(var_cols, colors)):
    ax.bar(periods_f, fevd_rt2[:, i] * 100,
           bottom=bottom, label=col, color=color, alpha=0.8)
    bottom += fevd_rt2[:, i] * 100

ax.set_title('FEVD -- Descompunerea variantei prognozei r_t^2 (volatilitate)')
ax.set_xlabel('Orizont (zile)')
ax.set_ylabel('Procent (%)')
ax.legend(loc='upper right')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_03_fevd.png', dpi=130, bbox_inches='tight')
plt.show()

print('FEVD la orizont 1, 5, 10 zile:')
for h in [0, 4, 9]:
    print(f'  Orizont h={h+1}:')
    for i, col in enumerate(var_cols):
        print(f"    {col:<20}: {fevd_rt2[h, i]*100:.1f}%")


---
## 4.7 DCC-GARCH bivariat

### Ce este DCC-GARCH

DCC (Dynamic Conditional Correlation) GARCH modeleaza cum **corelatia**
dintre doua serii se schimba in timp.

Pentru perechea (r_t, S_polarity_dm) ne intrebam:
*In perioadele de criza (COVID 2020, Ukraine 2022), corelatia dintre
randamente si sentiment devine mai puternica?*

Implementare in doua etape:
1. Estimam GARCH(1,1) univariat pe fiecare serie (filtram volatilitatea)
2. Modelam corelatia dinamica dintre reziduurile standardizate

### Interpretare

rho_t aproape de -1: sentiment negativ si volatilitate ridicata merg impreuna
rho_t aproape de 0:  sentiment si volatilitate sunt decuplate temporar
Variatia lui rho_t:  regimuri diferite (criza vs. calm)


In [ ]:
# ── DCC-GARCH (implementare manuala in 2 pasi) ────────────────────────
print("Estimare DCC-GARCH bivariat pe (r_t, S_polarity_dm)...")

s_train = train['S_polarity_dm'].dropna()
r_tr    = train['r_t'].dropna()

# Aliniem seriile
common_idx = r_tr.index.intersection(s_train.index)
r_aligned  = r_tr[common_idx]
s_aligned  = s_train[common_idx]

# Pas 1: GARCH(1,1) univariat pe fiecare serie
print("Pas 1: GARCH univariat pe r_t...")
garch_r = arch_model(r_aligned, vol='GARCH', p=1, q=1, dist='t')
fit_r   = garch_r.fit(disp='off')
std_resid_r = fit_r.std_resid.dropna()

print("Pas 1: GARCH univariat pe S_polarity_dm...")
# Scalat x100 pentru stabilitate numerica
garch_s = arch_model(s_aligned * 100, vol='GARCH', p=1, q=1, dist='t')
fit_s   = garch_s.fit(disp='off')
std_resid_s = fit_s.std_resid.dropna()

# Aliniem reziduurile standardizate
common_resid = std_resid_r.index.intersection(std_resid_s.index)
e1 = std_resid_r[common_resid].values
e2 = std_resid_s[common_resid].values

# Pas 2: DCC - estimam corelatia dinamica
# Parametrii DCC: alpha_dcc, beta_dcc
# Q_t = (1 - a - b)*Q_bar + a*(e_{t-1}*e_{t-1}') + b*Q_{t-1}
# R_t = diag(Q_t)^{-1/2} * Q_t * diag(Q_t)^{-1/2}

def dcc_filter(e1, e2, alpha=0.05, beta=0.90):
    n = len(e1)
    Q_bar = np.array([[1.0, np.corrcoef(e1, e2)[0,1]],
                       [np.corrcoef(e1, e2)[0,1], 1.0]])
    Q = Q_bar.copy()
    rho = np.zeros(n)

    for t in range(1, n):
        e_prev = np.array([[e1[t-1]**2, e1[t-1]*e2[t-1]],
                            [e1[t-1]*e2[t-1], e2[t-1]**2]])
        Q = (1 - alpha - beta) * Q_bar + alpha * e_prev + beta * Q
        q11 = max(Q[0,0], 1e-8)
        q22 = max(Q[1,1], 1e-8)
        rho[t] = Q[0,1] / np.sqrt(q11 * q22)
        rho[t] = np.clip(rho[t], -0.999, 0.999)

    return rho

rho_dynamic = dcc_filter(e1, e2, alpha=0.05, beta=0.90)

print(f"Correlatie medie: {rho_dynamic.mean():.4f}")
print(f"Correlatie min:   {rho_dynamic.min():.4f}")
print(f"Correlatie max:   {rho_dynamic.max():.4f}")


In [ ]:
# ── Plot DCC ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

rho_index = common_resid[:len(rho_dynamic)]

axes[0].plot(r_aligned.index, r_aligned.values,
             color='steelblue', linewidth=0.7, label='r_t')
axes[0].set_title('Log-randamente r_t')
axes[0].set_ylabel('%')

axes[1].plot(rho_index, rho_dynamic,
             color='darkgreen', linewidth=0.9, label='Coreltie dinamica rho_t')
axes[1].axhline(rho_dynamic.mean(), color='red', linewidth=0.8,
                linestyle='--', label=f'Media ({rho_dynamic.mean():.3f})')
axes[1].axhline(0, color='black', linewidth=0.4, linestyle='--')
axes[1].set_title('DCC-GARCH: Corelatie dinamica r_t x S_polarity_dm')
axes[1].set_ylabel('Corelatie conditionata')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-04-15'),
               alpha=0.15, color='red', label='COVID')
    ax.axvspan(pd.Timestamp('2022-02-24'), pd.Timestamp('2022-06-30'),
               alpha=0.15, color='orange', label='Ukraine')

plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_04_dcc_garch.png', dpi=130, bbox_inches='tight')
plt.show()

print("Interpretare:")
print("  Corelatie mai negativa in perioade de criza -> sentiment negativ merge cu volatilitate ridicata")
print("  Variatia in timp a corelatiei -> justifica DCC in loc de corelatie constanta")


---
## 4.8 GARCH-X -- Modelul central al proiectului

### Ecuatiile GARCH-X

Ecuatia medie:
r_t = mu + epsilon_t

Ecuatia variantei GARCH(1,1) clasic:
sigma_t^2 = omega + alpha * epsilon_{t-1}^2 + beta * sigma_{t-1}^2

Ecuatia variantei GARCH-X cu sentiment:
sigma_t^2 = omega + alpha * epsilon_{t-1}^2 + beta * sigma_{t-1}^2 + gamma * S_{t-1}^{polarity,dm}

Coeficientul **gamma** este raspunsul central:
- gamma < 0: sentiment negativ ieri -> volatilitate mai mare azi (intuitiv)
- gamma semnificativ statistic: sentimentul aduce informatie incrementala

### Specificatii testate

1. GARCH(1,1) pur -- benchmark
2. GARCH-X cu S_polarity_lag1 -- sentiment global
3. EGARCH-X cu S_polarity_lag1 -- cu asimetrie


In [ ]:
# ── Pregatim regresorii externi ───────────────────────────────────────
# S_polarity_lag1 e deja in dataset (calculat in Notebook 01)
# Il demenam folosind media din train

s_lag1_mean = train['S_polarity_lag1'].mean()
train['S_polarity_lag1_dm'] = train['S_polarity_lag1'] - s_lag1_mean
test['S_polarity_lag1_dm']  = test['S_polarity_lag1']  - s_lag1_mean

# Alinierea: regresorii externi trebuie sa aiba acelasi index ca r_t
r_tr_clean = train['r_t'].dropna()
x_train    = train['S_polarity_lag1_dm'].reindex(r_tr_clean.index).fillna(0)

print(f"r_train: {len(r_tr_clean)} obs")
print(f"x_train (S_lag1_dm): {len(x_train)} obs")
print(f"Verificare aliniere: {(r_tr_clean.index == x_train.index).all()}")
print(f"Statistici S_polarity_lag1_dm (train):")
print(x_train.describe().round(4))


In [ ]:
# ── GARCH(1,1) BASELINE ──────────────────────────────────────────────
print("Estimare GARCH(1,1) baseline...")
garch_base = arch_model(r_tr_clean, vol='GARCH', p=1, q=1, dist='t')
fit_base   = garch_base.fit(disp='off')
print(fit_base.summary())


In [ ]:
# ── GARCH-X cu S_polarity_lag1_dm ────────────────────────────────────
print("Estimare GARCH-X (sentiment in ecuatia variantei)...")

garch_x = arch_model(r_tr_clean, x=x_train.values.reshape(-1,1),
                      vol='GARCH', p=1, q=1, dist='t')
fit_x = garch_x.fit(disp='off')
print(fit_x.summary())


In [ ]:
# ── Tabel comparativ GARCH vs GARCH-X ────────────────────────────────
print("=== COMPARATIE GARCH vs GARCH-X ===")
print(f"{'Model':<20} {'AIC':>10} {'BIC':>10} {'LogLik':>10}")
print("-" * 54)
print(f"  {'GARCH(1,1)':<18} {fit_base.aic:10.2f} {fit_base.bic:10.2f} {fit_base.loglikelihood:10.2f}")
print(f"  {'GARCH-X':<18} {fit_x.aic:10.2f} {fit_x.bic:10.2f} {fit_x.loglikelihood:10.2f}")
print()

# Coeficientul gamma (sentiment)
gamma_val = fit_x.params.get('x0', np.nan)
gamma_pval = fit_x.pvalues.get('x0', np.nan)

print(f"Coeficient gamma (sentiment): {gamma_val:.6f}")
print(f"p-value gamma:                {gamma_pval:.4f}")
print()
if gamma_pval < 0.05:
    print("-> gamma semnificativ la 5% -> sentimentul aduce informatie incrementala!")
    if gamma_val < 0:
        print("-> gamma < 0: sentiment negativ -> volatilitate mai mare (intuitiv corect)")
    else:
        print("-> gamma > 0: sentiment negativ -> volatilitate mai mica (surprinzator)")
elif gamma_pval < 0.10:
    print("-> gamma marginal semnificativ la 10%")
else:
    print("-> gamma nesemnificativ -> sentimentul nu aduce informatie incrementala in varinta")
    print("   (desi poate contribui prin medie -- verificam ecuatia medie)")


In [ ]:
# ── Forecast GARCH-X pe test 2024 ────────────────────────────────────
print("Forecast GARCH-X pe test 2024...")

r_test_clean  = test['r_t'].dropna()
x_test        = test['S_polarity_lag1_dm'].reindex(r_test_clean.index).fillna(0)
n_test        = len(r_test_clean)
actual_var    = r_test_clean.values**2

# Metrici utilitare
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

def mae_fn(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def qlike(sigma2_true, h_pred):
    h = np.maximum(h_pred, 1e-8)
    s = np.maximum(sigma2_true, 1e-8)
    return float(np.mean(s / h - np.log(s / h) - 1))

# Forecast GARCH baseline
fc_base = fit_base.forecast(horizon=n_test, reindex=False)
var_base = fc_base.variance.values
if var_base.ndim == 2:
    var_base = var_base[-1]
var_base = var_base[:n_test]

# Forecast GARCH-X
# GARCH-X forecast: trebuie sa pasam x-ul din test
try:
    fc_x = fit_x.forecast(horizon=n_test,
                            x=x_test.values.reshape(-1,1)[:n_test],
                            reindex=False)
    var_x = fc_x.variance.values
    if var_x.ndim == 2:
        var_x = var_x[-1]
    var_x = var_x[:n_test]
    garch_x_ok = True
except Exception as e:
    print(f"GARCH-X forecast cu x a esuat: {e}")
    print("Folosim forecast GARCH-X fara x nou (conservativ)")
    fc_x = fit_x.forecast(horizon=n_test, reindex=False)
    var_x = fc_x.variance.values
    if var_x.ndim == 2:
        var_x = var_x[-1]
    var_x = var_x[:n_test]
    garch_x_ok = False

n = min(len(actual_var), len(var_base), len(var_x))
results_vol = {
    'GARCH(1,1)_baseline': {
        'RMSE': rmse(actual_var[:n], var_base[:n]),
        'MAE':  mae_fn(actual_var[:n], var_base[:n]),
        'QLIKE': qlike(actual_var[:n], var_base[:n])
    },
    'GARCH-X_sentiment': {
        'RMSE': rmse(actual_var[:n], var_x[:n]),
        'MAE':  mae_fn(actual_var[:n], var_x[:n]),
        'QLIKE': qlike(actual_var[:n], var_x[:n])
    }
}

print(f"{'Model':<25} {'RMSE':>8} {'MAE':>8} {'QLIKE':>8}")
print("-" * 54)
for name, m in results_vol.items():
    print(f"  {name:<23} {m['RMSE']:8.4f} {m['MAE']:8.4f} {m['QLIKE']:8.4f}")


In [ ]:
# ── Plot GARCH vs GARCH-X ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(r_test_clean.index[:n], actual_var[:n],
        color='gray', linewidth=0.5, alpha=0.6, label='r_t^2 actual', zorder=1)
ax.plot(r_test_clean.index[:n], var_base[:n],
        color='steelblue', linewidth=1.2, label='GARCH(1,1) baseline', zorder=2)
ax.plot(r_test_clean.index[:n], var_x[:n],
        color='crimson', linewidth=1.2, linestyle='--',
        label='GARCH-X (cu sentiment)', zorder=3)

ax.set_title('Forecast varianta conditionata: GARCH vs GARCH-X (Test 2024)')
ax.set_ylabel('Varianta (%^2)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_05_garch_x_forecast.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Volatilitate conditionata GARCH-X pe train ───────────────────────
# Vizualizam cum capteaza GARCH-X perioadele de criza
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(r_tr_clean.index, r_tr_clean.values,
             color='steelblue', linewidth=0.7)
axes[0].set_title('r_t -- Log-randamente (train)')
axes[0].set_ylabel('%')

axes[1].plot(r_tr_clean.index, fit_base.conditional_volatility,
             color='steelblue', linewidth=0.9, label='GARCH(1,1) sigma_t')
axes[1].plot(r_tr_clean.index, fit_x.conditional_volatility,
             color='crimson', linewidth=0.9, linestyle='--',
             label='GARCH-X sigma_t')
axes[1].set_title('Volatilitate conditionata: GARCH vs GARCH-X (train)')
axes[1].set_ylabel('Sigma_t (%)')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-04-15'),
               alpha=0.12, color='red')
    ax.axvspan(pd.Timestamp('2022-02-24'), pd.Timestamp('2022-06-30'),
               alpha=0.12, color='orange')

plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_06_garch_x_train.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 4.9 Comparatie finala: univariat vs. multivariat

### Tabel sintetic

Comparam cele mai bune modele din Notebook 02 (univariate) cu modelele
multivariate din acest notebook.

### Testul Diebold-Mariano

Verificam daca imbunatatirea GARCH-X fata de GARCH(1,1) este
statistic semnificativa.


In [ ]:
# ── Tabel comparativ final ────────────────────────────────────────────
print("=== COMPARATIE FINALA: UNIVARIAT vs. MULTIVARIAT ===")

# Rezultate din Notebook 02 (hardcoded sau incarcate din CSV)
try:
    nb02_results = pd.read_csv(FIG_PATH / 'nb02_results.csv', index_col=0)
    print("Rezultate Notebook 02 incarcate din CSV:")
    print(nb02_results.round(4))
    print()
except FileNotFoundError:
    print("Fisierul nb02_results.csv nu a fost gasit.")
    print("Ruleaza mai intai Notebook 02 complet.")
    nb02_results = None

# Rezultate Notebook 03
print("=== Rezultate Notebook 03 (modele multivariate) ===")
print(f"{'Model':<25} {'RMSE':>8} {'MAE':>8} {'QLIKE':>8}")
print("-" * 54)
for name, m in results_vol.items():
    print(f"  {name:<23} {m['RMSE']:8.4f} {m['MAE']:8.4f} {m['QLIKE']:8.4f}")


In [ ]:
# ── Diebold-Mariano: GARCH-X vs GARCH(1,1) ──────────────────────────
def diebold_mariano(actual, pred1, pred2, loss='mse'):
    if loss == 'mse':
        d = (actual - pred1)**2 - (actual - pred2)**2
    else:
        d = np.abs(actual - pred1) - np.abs(actual - pred2)
    n       = len(d)
    d_mean  = d.mean()
    var_d   = np.var(d, ddof=1) / n
    dm_stat = d_mean / np.sqrt(max(var_d, 1e-12))
    p_val   = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return dm_stat, p_val

print("=== TEST DIEBOLD-MARIANO: GARCH-X vs GARCH(1,1) ===")
n_dm = min(len(actual_var), len(var_base), len(var_x))

dm_mse,  p_mse  = diebold_mariano(actual_var[:n_dm], var_x[:n_dm], var_base[:n_dm], 'mse')
dm_mae,  p_mae  = diebold_mariano(actual_var[:n_dm], var_x[:n_dm], var_base[:n_dm], 'mae')

print(f"{'Loss':<8} {'DM stat':>10} {'p-value':>10} {'Concluzie':>30}")
print("-" * 63)
for loss_name, dm, p in [('MSE', dm_mse, p_mse), ('MAE', dm_mae, p_mae)]:
    if p < 0.05 and dm < 0:
        concl = 'GARCH-X mai bun (5%) **'
    elif p < 0.10 and dm < 0:
        concl = 'GARCH-X mai bun (10%) *'
    elif p < 0.05 and dm > 0:
        concl = 'GARCH baseline mai bun'
    else:
        concl = 'Fara diferenta semnificativa'
    print(f"  {loss_name:<6} {dm:10.3f} {p:10.4f} {concl:>30}")

print()
print("Interpretare:")
print("  DM < 0 si p < 0.05: GARCH-X bate GARCH(1,1) -> sentimentul aduce informatie")
print("  DM > 0 si p < 0.05: GARCH(1,1) bate GARCH-X -> sentimentul nu ajuta")
print("  p > 0.05: nu putem distinge statistic intre modele")


In [ ]:
# ── Plot final sintetic ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: volatilitate conditionata in perioadele de criza
axes[0].plot(r_test_clean.index[:n_dm], actual_var[:n_dm],
             color='gray', linewidth=0.5, alpha=0.6, label='r_t^2 actual')
axes[0].plot(r_test_clean.index[:n_dm], var_base[:n_dm],
             color='steelblue', linewidth=1.2, label='GARCH(1,1)')
axes[0].plot(r_test_clean.index[:n_dm], var_x[:n_dm],
             color='crimson', linewidth=1.2, linestyle='--', label='GARCH-X')
axes[0].set_title('Forecast varianta: GARCH vs GARCH-X (Test 2024)')
axes[0].set_ylabel('Varianta (%^2)')
axes[0].legend(fontsize=9)

# Panel 2: diferenta de forecast (GARCH-X - GARCH) -- cand ajuta sentimentul?
diff = var_x[:n_dm] - var_base[:n_dm]
axes[1].bar(r_test_clean.index[:n_dm], diff,
            color=['crimson' if d > 0 else 'steelblue' for d in diff],
            alpha=0.7, linewidth=0)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Diferenta forecast: GARCH-X minus GARCH(1,1)')
axes[1].set_ylabel('Diferenta varianta')

plt.tight_layout()
plt.savefig(FIG_PATH / 'nb03_07_final_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

print("Interpretare panel 2:")
print("  Bare rosii: GARCH-X prezice volatilitate mai mare decat GARCH")
print("  Bare albastre: GARCH-X prezice volatilitate mai mica decat GARCH")
print("  Daca bare rosii coincid cu zile de volatilitate ridicata -> GARCH-X e mai calibrat")


---
## Concluzii Notebook 03

### Ce am testat

1. **Cointegrare Johansen** -- daca variabilele au relatie de echilibru pe termen lung
2. **VAR(p)** -- dinamica multivariata a sistemului [r_t, r_t_sq, S_polarity]
3. **Granger causality** -- directia de predictibilitate intre sentiment si piata
4. **IRF + FEVD** -- impactul unui soc de sentiment asupra volatilitatii
5. **DCC-GARCH** -- corelatia dinamica intre randamente si sentiment
6. **GARCH-X** -- sentimentul direct in ecuatia de varianta

### Raspunsul la intrebarea centrala

Coeficientul gamma din GARCH-X si testul Diebold-Mariano ne dau raspunsul empiric:

- **Daca gamma < 0 si semnificativ**: sentimentul negativ prezice volatilitate ridicata,
  deci presa financiara aduce informatie incrementala peste modelele clasice
- **Daca DM test respinge H0 in favoarea GARCH-X**: imbunatatirea e statistic semnificativa,
  nu doar numerica

### Limitari metodologice

1. FinBERT e antrenat pe alte texte decat NYT -- posibil bias in scoring
2. Sentimentul captureaza mai ales tonul, nu continutul semantic al stirilor
3. Endogeneitate: piata misca sentimentul si sentimentul misca piata simultan
4. GARCH-X assume linearitate in efectul sentimentului -- posibil efect neliniar
   (sentimentul extrem de negativ poate amplifica mai mult decat cel moderat)

### Ce ar urma (daca extensii ar fi necesare)

- LSTM cu sentiment (replica Kim et al. 2023) -- comparatie deep learning
- Analiza pe subperioade (COVID vs. non-COVID) -- stabilitatea efectului
- Sentiment per news_desk (Business vs. DealBook vs. Washington) -- care desk prezice mai bine
